# Retrain Joint mBERT (en/es/hi/te) — seeds 42, 123, 7

**Why this exists:** the `joint_mbert` checkpoints for `en_es_hi_te` (seeds 42/123/7) lost their fine-tuned BERT encoder weights at some point — only `task_heads.pt` (the 3 small linear heads) survived, everywhere checked. Running inference with those alone gives chance-level results (verified: 0.51 cls accuracy on English, the model's own training language). This notebook retrains all 3 seeds via `training/Train_Join.py`, **persists the full encoder** (`model.safetensors`) to Drive this time, then runs the eval directly here: a sanity check against the historical 5-language test set, and the real N2 French/German zero-shot pass.

T4 is fine (matches the original 9-system taxonomy's runtime, mBERT-base is small). Each seed should take well under an hour.

**Do not run any training cell until Cell 4's gate prints `islink: True` for both `models` and `results`** — this repo commits real `models/`+`results/` dirs, so a naive symlink nests instead of replacing, and output silently goes to ephemeral `/content` (this has cost 3-4h retrains twice before).

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
SEEDS    = [42, 123, 7]
DRIVE_ROOT = '/content/drive/MyDrive/Idiomator_Research'
OUT_DIRS = {42: 'models/en_es_hi_te/joint_mbert', 123: 'models/main_s123/joint_mbert', 7: 'models/main_s7/joint_mbert'}
print('repo:', REPO, '| seeds:', SEEDS)

In [ ]:
# 2. Fresh clone — always start from a clean clone, never resume a possibly-stale one
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys, shutil
from pathlib import Path
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
print('cwd:', os.getcwd())
print('head:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
assert Path('training/Train_Join.py').exists(), 'Train_Join.py missing — wrong repo/branch?'
assert Path('data/idioms_structured/Splits/test_heldout_fr_de.jsonl').exists(), 'fr/de test set missing — push it before running'

In [ ]:
# 3. Symlink models/ + results/ to Drive — HARD GATE, do not proceed if this fails.
# The repo commits real models/+results/ dirs (54 files); a naive `ln -sfn` would
# nest instead of replace, so output silently lands in ephemeral /content.
for d in ['models', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
    p = os.path.join(REPO, d)
    if os.path.islink(p):
        os.unlink(p)
    elif os.path.isdir(p):
        shutil.rmtree(p)
    os.symlink(f'{DRIVE_ROOT}/{d}', p)

ok = True
for d in ['models', 'results']:
    p = os.path.join(REPO, d)
    is_link = os.path.islink(p)
    ok &= is_link
    print(d, 'islink:', is_link, '->', os.readlink(p) if is_link else '(REAL DIR - BAD)')
assert ok, 'Symlinks NOT set up — STOP, do not run training cells.'
print('\nGate passed. Safe to proceed.')

In [ ]:
# 4. Install deps + confirm GPU (T4 is fine)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 5. Smoke test (~1-2 min): 1 epoch, English only, throwaway dir — catches path/CLI bugs before real spend
!python training/Train_Join.py \
    --langs English --output_dir /tmp/_joint_smoke --epochs 1 --batch_size 8 --seed 42

In [ ]:
# 6. FULL RUNS — 3 seeds, en/es/hi/te, output straight to Drive-symlinked models/.
#    Re-run this cell after any disconnect; each seed's output_dir is separate so finished ones are untouched.
for seed in SEEDS:
    out_dir = OUT_DIRS[seed]
    print(f'\n=== seed {seed} -> {out_dir} ===')
    !python training/Train_Join.py --langs English Spanish Hindi Telugu \
        --test_langs English Spanish Hindi Telugu Indonesian \
        --output_dir {out_dir} --seed {seed}

In [ ]:
# 7. Persistence check — confirms the encoder actually landed on Drive this time (the thing that got lost before).
# Read back from the Drive path itself, not /content, so a fresh mount would still see it.
for seed, out_dir in OUT_DIRS.items():
    drive_best = Path(DRIVE_ROOT) / out_dir / 'best_model'
    has_encoder = (drive_best / 'model.safetensors').exists() or (drive_best / 'pytorch_model.bin').exists()
    has_heads   = (drive_best / 'task_heads.pt').exists()
    flag = 'OK' if has_encoder and has_heads else 'MISSING FILES — do not treat as done, do not run eval cells'
    print(f'seed {seed:<4} encoder={has_encoder} heads={has_heads}  [{flag}]')

In [ ]:
# 8. Sanity-check eval — historical 5-language test set (English/Spanish/Hindi/Telugu/Indonesian).
# Numbers should land near the historical range (~0.66-0.80 cls accuracy per language) —
# if any seed comes back near-chance (~0.5), that seed's retrain didn't actually work, stop and debug before trusting fr/de.
# Renames output immediately — Cell 9 writes the same filename for fr/de, don't let it clobber this one.
import json, collections
for seed, out_dir in OUT_DIRS.items():
    print(f'\n=== seed {seed} — historical test.jsonl sanity check ===')
    !python Evaluation/infer_shared_test.py \
        --test_path data/idioms_structured/Splits/test.jsonl \
        --joint_dir {out_dir} --joint_only
    raw_out = Path(out_dir) / 'test_predictions_shared.jsonl'
    sanity_out = Path(out_dir) / 'test_predictions_shared_5lang_sanity.jsonl'
    shutil.copy(raw_out, sanity_out)
    preds = [json.loads(l) for l in open(sanity_out)]
    by_lang = collections.defaultdict(list)
    for r in preds:
        by_lang[r['language']].append(r)
    for lang, rs in sorted(by_lang.items()):
        acc = sum(r['correct'] for r in rs) / len(rs)
        print(f'  {lang:<12} n={len(rs):<5} cls_acc={acc:.4f}' + ('  <-- near-chance, investigate' if acc < 0.58 else ''))

In [ ]:
# 9. The actual N2 eval — French/German zero-shot (the point of this whole retrain).
# Only trust this if Cell 8 came back in the historical range for all 3 seeds.
for seed, out_dir in OUT_DIRS.items():
    print(f'\n=== seed {seed} — French/German zero-shot ===')
    !python Evaluation/infer_shared_test.py \
        --test_path data/idioms_structured/Splits/test_heldout_fr_de.jsonl \
        --joint_dir {out_dir} --joint_only
    raw_out = Path(out_dir) / 'test_predictions_shared.jsonl'
    frde_out = Path(out_dir) / 'test_predictions_shared_frde.jsonl'
    shutil.copy(raw_out, frde_out)
    preds = [json.loads(l) for l in open(frde_out)]
    by_lang = collections.defaultdict(list)
    for r in preds:
        by_lang[r['language']].append(r)
    for lang, rs in sorted(by_lang.items()):
        acc = sum(r['correct'] for r in rs) / len(rs)
        overlap = sum(r['overlap_f1'] for r in rs) / len(rs)
        print(f'  {lang:<12} n={len(rs):<5} cls_acc={acc:.4f}  span_overlap_f1={overlap:.4f}')
print('\nDone. fr/de predictions saved as test_predictions_shared_frde.jsonl in each seed\'s joint_mbert/ dir (on Drive).')
print('Come back to the main session with these numbers for the Holm correction + GPT-4o comparison.')